# Getting all available Swiss supermarkets using the overpass Web API

## Libraries and settings

In [13]:
# Libraries
import os
import requests
import json
import folium
from pandas import json_normalize

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

/workspaces/data_analytics/Week_01


## Overpass turbo query to get all available supermarkets in Switzerland

In [14]:
# Overpass API URL
URL = 'http://overpass-api.de/api/interpreter'

# Overpass turbo query
QUERY = """
        [out:json];
        area["ISO3166-1"="CH"][admin_level=2];
        node ["shop"="supermarket"](area);
        out;"""

# Web API request
r = requests.get(
    URL,
    params={'data': QUERY}, 
    headers={'User-Agent': 'my-app/1.0'},
    timeout=30
)
r.raise_for_status()
data = r.json()['elements']

# Save data to file
with open('supermarkets.json', 'w', encoding="utf-8") as json_file:
    json.dump(data, json_file)

# Store data in data frame
df = json_normalize(data)

# Number of rows and columns
print(df.shape)

# First rows
df.head()

(3479, 300)


,type,id,lat,lon,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.brand,...,tags.diet:organic,tags.diet:seafood,tags.recycling:low_energy_bulbs,tags.opening_hours:cafe,tags.payment:discover_card,tags.name:zh-Hans,tags.panoramax,tags.ramp:wheelchair,tags.wheelchair:description:de,tags.wheelchair:description:en
0,node,33126515,47.155616,9.037915,Schänis,CH,32,8718,Biltnerstrasse,Spar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,36726161,47.228183,8.965330,Uznach,NaN,4,8730,Wiesentalstrasse,Migros,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,39768209,47.225154,8.969868,Uznach,NaN,NaN,8730,NaN,Coop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,39947904,47.376732,8.542161,Zürich,CH,1,8001,Bahnhofbrücke,Coop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,48932835,47.375020,8.522895,Zürich,NaN,7,8004,Wengistrasse,Migros,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Plot supermarkets on map

In [15]:
# Subset of supermarkets by brand
locations = df[["lat", "lon", "tags.brand", "tags.shop"]].loc[df["tags.brand"].isin(['Migros', 'Coop'])]
print(locations.head(5))

# Create map
map = folium.Map(location=[locations.lat.mean(), 
                           locations.lon.mean()], 
                 zoom_start=8, 
                 control_scale=True)

# Add maker symbols
for index, location_info in locations.iterrows():
    folium.Marker([location_info["lat"], 
                   location_info["lon"]], 
                  popup=location_info["tags.brand"]).add_to(map)

# Plot map
map
map.save("supermarkets_map.html")

         lat       lon tags.brand    tags.shop
1  47.228183  8.965330     Migros  supermarket
2  47.225154  8.969868       Coop  supermarket
3  47.376732  8.542161       Coop  supermarket
4  47.375020  8.522895     Migros  supermarket
5  47.491874  8.706448     Migros  supermarket


### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [16]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-09-22 09:09:53
Python Version: 3.11.16
-----------------------------------
